In [ ]:
import os
import math
import random
import warnings
from itertools import islice

import numpy as np
import pandas as pd
import yfinance as yf
import pywt
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import cvxpy as cp

from scipy.stats import skew, kurtosis, jarque_bera
from matplotlib.backends.backend_pdf import PdfPages
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8-darkgrid")

START_DATE = "2010-01-01"
END_DATE   = "2025-01-01"

WAVELET = "db8"
LEVELS  = 3
SCALES  = [1, 2, 3]  # 1 = short (D1), 2 = medium (D2), 3 = long (D3)

QUANTILES            = np.linspace(0.05, 0.95, 25)
CRASH_CUTOFF         = 0.10
SAFE_HAVEN_THRESHOLD = -0.01

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GRAMS_PER_OUNCE = 31.1034768
N_JOBS = -1   # use all available CPU cores

STOCKS = [
    "^NSEI",
    "RELIANCE.NS","TCS.NS","INFY.NS","HDFCBANK.NS","ICICIBANK.NS",
    "AXISBANK.NS","SBIN.NS","KOTAKBANK.NS","LT.NS","ITC.NS",
    "HINDUNILVR.NS","ASIANPAINT.NS",
    "MARUTI.NS","M&M.NS","TATAMOTORS.NS",
    "SUNPHARMA.NS","DRREDDY.NS","CIPLA.NS","DIVISLAB.NS",
    "ULTRACEMCO.NS","GRASIM.NS","TATASTEEL.NS","JSWSTEEL.NS",
    "HINDALCO.NS","NTPC.NS","POWERGRID.NS","ONGC.NS",
    "COALINDIA.NS","BHARTIARTL.NS","ADANIENT.NS","ADANIPORTS.NS",
    "BAJFINANCE.NS","BAJAJFINSV.NS","HCLTECH.NS","WIPRO.NS",
    "TECHM.NS","NESTLEIND.NS","BRITANNIA.NS","TITAN.NS",
    "APOLLOHOSP.NS"
]

sector_map = {
    "RELIANCE.NS"  : "Energy",    "TCS.NS"       : "IT",
    "INFY.NS"      : "IT",        "HDFCBANK.NS"  : "Banking",
    "ICICIBANK.NS" : "Banking",   "AXISBANK.NS"  : "Banking",
    "SBIN.NS"      : "Banking",   "KOTAKBANK.NS" : "Banking",
    "LT.NS"        : "Infrastructure", "ITC.NS"  : "FMCG",
    "HINDUNILVR.NS": "FMCG",      "NESTLEIND.NS" : "FMCG",
    "BRITANNIA.NS" : "FMCG",      "ASIANPAINT.NS": "Consumer",
    "TITAN.NS"     : "Consumer",  "MARUTI.NS"    : "Auto",
    "M&M.NS"       : "Auto",      "TATAMOTORS.NS": "Auto",
    "SUNPHARMA.NS" : "Pharma",    "DRREDDY.NS"   : "Pharma",
    "CIPLA.NS"     : "Pharma",    "DIVISLAB.NS"  : "Pharma",
    "ULTRACEMCO.NS": "Cement",    "GRASIM.NS"    : "Cement",
    "TATASTEEL.NS" : "Metals",    "JSWSTEEL.NS"  : "Metals",
    "HINDALCO.NS"  : "Metals",    "NTPC.NS"      : "Power",
    "POWERGRID.NS" : "Power",     "ONGC.NS"      : "Energy",
    "COALINDIA.NS" : "Energy",    "BHARTIARTL.NS": "Telecom",
    "ADANIENT.NS"  : "Conglomerate","ADANIPORTS.NS": "Logistics",
    "BAJFINANCE.NS": "Financials","BAJAJFINSV.NS": "Financials",
    "HCLTECH.NS"   : "IT",        "WIPRO.NS"     : "IT",
    "TECHM.NS"     : "IT",        "APOLLOHOSP.NS": "Healthcare",
    "^NSEI"        : "Market_Index"
}

ticker_to_code = {
    "RELIANCE.NS"  : "RIL",   "TCS.NS"       : "TCS",
    "INFY.NS"      : "INFY",  "HDFCBANK.NS"  : "HDFC",
    "ICICIBANK.NS" : "ICICI", "AXISBANK.NS"  : "AXIS",
    "SBIN.NS"      : "SBIN",  "KOTAKBANK.NS" : "KOTAK",
    "LT.NS"        : "LT",    "ITC.NS"       : "ITC",
    "HINDUNILVR.NS": "HUL",   "ASIANPAINT.NS": "ASPL",
    "MARUTI.NS"    : "MSIL",  "M&M.NS"       : "MML",
    "TATAMOTORS.NS": "TML",   "SUNPHARMA.NS" : "SUNP",
    "DRREDDY.NS"   : "DRDY",  "CIPLA.NS"     : "CIPLA",
    "DIVISLAB.NS"  : "DIVS",  "ULTRACEMCO.NS": "ULCEM",
    "GRASIM.NS"    : "GRM",   "TATASTEEL.NS" : "TSL",
    "JSWSTEEL.NS"  : "JSL",   "HINDALCO.NS"  : "HNDC",
    "NTPC.NS"      : "NTPC",  "POWERGRID.NS" : "PGRD",
    "ONGC.NS"      : "ONGC",  "COALINDIA.NS" : "COAL",
    "BHARTIARTL.NS": "ARTL",  "ADANIENT.NS"  : "ADANI",
    "ADANIPORTS.NS": "ADPT",  "BAJFINANCE.NS": "BFL",
    "BAJAJFINSV.NS": "BJFSV", "HCLTECH.NS"   : "HCL",
    "WIPRO.NS"     : "WIPRO", "TECHM.NS"     : "TECHM",
    "NESTLEIND.NS" : "NESTL", "BRITANNIA.NS" : "BRIT",
    "TITAN.NS"     : "TITAN", "APOLLOHOSP.NS": "APOL",
    "^NSEI"        : "NSEI",  "Gold"         : "Gold",
}


In [ ]:
def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

def extract_close_from_download(raw, ticker_label=None):
    if raw is None or raw.empty:
        return pd.DataFrame()
    if isinstance(raw.columns, pd.MultiIndex):
        level0 = raw.columns.get_level_values(0)
        level1 = raw.columns.get_level_values(1)
        if "Close" in level1:
            return raw.xs("Close", axis=1, level=1)
        if "Close" in level0:
            return raw.xs("Close", axis=1, level=0)
        return pd.DataFrame()
    if "Close" in raw.columns:
        if ticker_label is None:
            ticker_label = "Close"
        return raw[["Close"]].rename(columns={"Close": ticker_label})
    return pd.DataFrame()

def download_stock_closes(tickers, start, end, batch_size=12):
    frames = []
    tickers = list(tickers)
    for batch in chunked(tickers, batch_size):
        try:
            raw = yf.download(
                batch, start=start, end=end,
                group_by="ticker", auto_adjust=True,
                progress=False, threads=False
            )
            close = extract_close_from_download(raw)
            if not close.empty:
                frames.append(close)
        except Exception as e:
            print(f"Warning: batch download failed for {batch}: {e}")
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, axis=1)
    out = out.loc[:, ~out.columns.duplicated()]
    return out.sort_index()

def download_single_close(ticker, start, end, label=None):
    raw = yf.download(
        ticker, start=start, end=end,
        auto_adjust=True, progress=False, threads=False
    )
    close = extract_close_from_download(raw, ticker_label=label or ticker)
    if close.empty:
        return pd.DataFrame()
    if isinstance(close, pd.Series):
        close = close.to_frame(name=label or ticker)
    return close.sort_index()

def safe_stats(series):
    s = pd.Series(series).dropna()
    if len(s) < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
    try:
        jb_stat, jb_p = jarque_bera(s)
    except Exception:
        jb_stat, jb_p = np.nan, np.nan
    return s.mean(), s.std(), skew(s), kurtosis(s), jb_stat, jb_p

def wavelet_mra_components(series, wavelet=WAVELET, level=LEVELS):
    """Returns detail components D1/D2/D3 and approximation A."""
    x = np.asarray(series, dtype=float)
    coeffs = pywt.wavedec(x, wavelet, level=level)
    out = {}
    for detail_level in range(1, level + 1):
        coeffs_i = [np.zeros_like(c) for c in coeffs]
        coeff_index = level - detail_level + 1
        coeffs_i[coeff_index] = coeffs[coeff_index]
        rec = pywt.waverec(coeffs_i, wavelet)[:len(x)]
        out[detail_level] = rec
    approx_coeffs = [np.zeros_like(c) for c in coeffs]
    approx_coeffs[0] = coeffs[0]
    out["A"] = pywt.waverec(approx_coeffs, wavelet)[:len(x)]
    return out

def classify_crash_beta(crash_beta, threshold=SAFE_HAVEN_THRESHOLD):
    if np.isnan(crash_beta):
        return "NA"
    if crash_beta < threshold:
        return "SAFE HAVEN"
    if -abs(threshold) <= crash_beta <= abs(threshold):
        return "HEDGE"
    return "CO-MOVEMENT"

def gaussian_kernel(u):
    return np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)

def solve_weighted_quantile_regression(x, y, weights, theta):
    """Weighted quantile regression via cvxpy."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(weights, dtype=float)
    n = len(x)
    if n == 0:
        return np.nan
    w = np.maximum(w, 1e-12)
    w = w / w.sum()
    alpha   = cp.Variable()
    beta    = cp.Variable()
    u_plus  = cp.Variable(n, nonneg=True)
    u_minus = cp.Variable(n, nonneg=True)
    constraints = [y - alpha - beta * x == u_plus - u_minus]
    objective   = cp.Minimize(
        cp.sum(cp.multiply(w, theta * u_plus + (1 - theta) * u_minus))
    )
    problem = cp.Problem(objective, constraints)
    for solver in ["CLARABEL", "ECOS", "SCS"]:
        try:
            problem.solve(solver=solver, warm_start=True, verbose=False)
            if beta.value is not None and np.isfinite(beta.value):
                return float(beta.value)
        except Exception:
            continue
    return np.nan

def kernel_qq_matrix(stock_series, gold_series, quantiles=QUANTILES, bandwidth=None):
    """True kernel-weighted QQ for one asset."""
    x_raw = np.asarray(stock_series, dtype=float)
    y_raw = np.asarray(gold_series,  dtype=float)
    n = len(x_raw)
    if n < 10:
        return np.full((len(quantiles), len(quantiles)), np.nan)
    if bandwidth is None:
        iqr   = np.subtract(*np.percentile(x_raw, [75, 25]))
        sigma = np.std(x_raw, ddof=1)
        scale = min(sigma, iqr / 1.34) if iqr > 0 else sigma
        bandwidth = 0.9 * scale * (n ** (-1/5))
        if (not np.isfinite(bandwidth)) or bandwidth <= 0:
            bandwidth = (sigma if sigma > 0 else 1.0) * (n ** (-1/5)) + 1e-8
    beta_matrix  = np.full((len(quantiles), len(quantiles)), np.nan)
    tau_q_values = {tau: np.quantile(x_raw, tau) for tau in quantiles}
    for j, tau in enumerate(quantiles):
        x_q     = tau_q_values[tau]
        x       = x_raw - x_q
        weights = gaussian_kernel((x_raw - x_q) / bandwidth)
        weights = np.maximum(weights, 1e-12)
        weights = weights / weights.sum()
        for i, theta in enumerate(quantiles):
            beta_matrix[i, j] = solve_weighted_quantile_regression(
                x, y_raw, weights, theta
            )
    return beta_matrix

def plot_heatmap(beta_matrix, title,
                 xlabel="Stock Quantile (\u03c4)",
                 ylabel="Gold Quantile (\u03b8)",
                 save_path=None):
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        beta_matrix,
        xticklabels=np.round(QUANTILES, 2),
        yticklabels=np.round(QUANTILES, 2),
        cmap="coolwarm", center=0, ax=ax
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig

def plot_mst_from_distance_matrix(dist_df, title,
                                   highlight_node="Gold",
                                   save_path=None,
                                   label_map=None):
    G = nx.Graph()
    nodes = list(dist_df.columns)
    for i in nodes:
        for j in nodes:
            if i != j:
                G.add_edge(i, j, weight=float(dist_df.loc[i, j]))
    mst = nx.minimum_spanning_tree(G, weight="weight")

    if label_map is None:
        label_map = {n: n for n in mst.nodes()}
    labels = {n: label_map.get(n, n) for n in mst.nodes()}

    node_sizes  = [700 if n == highlight_node else 500 for n in mst.nodes()]
    node_colors = ["#FF6B35" if n == highlight_node else "#4A90D9" for n in mst.nodes()]

    fig, ax = plt.subplots(figsize=(14, 10))
    pos = nx.kamada_kawai_layout(mst, weight="weight")

    nx.draw_networkx_edges(mst, pos, ax=ax, alpha=0.7, edge_color="black", width=1.5)
    nx.draw_networkx_nodes(mst, pos, ax=ax,
                           node_size=node_sizes,
                           node_color=node_colors,
                           alpha=0.92)
    for node, (x, y) in pos.items():
        lbl   = labels[node]
        fw    = "bold" if node == highlight_node else "normal"
        fsize = max(5, 9 - max(0, len(lbl) - 4))
        ax.text(x, y, lbl, ha="center", va="center",
                fontsize=fsize, fontweight=fw, color="black",
                clip_on=True)

    ax.axis("off")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=1000, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return mst


In [ ]:

_KAGGLE_INPUT = "/kaggle/input"
_PRELOADED    = False

def _find_kaggle_csv(name):
    """Search /kaggle/input recursively for a file matching name."""
    for root, dirs, files in os.walk(_KAGGLE_INPUT):
        for f in files:
            if f == name:
                return os.path.join(root, f)
    return None

_stocks_csv  = _find_kaggle_csv("Stocks_Prices_2010_2025.csv")
_gold_csv    = _find_kaggle_csv("Gold_Price_2010_2025.csv")
_returns_csv = _find_kaggle_csv("Returns_Aligned_2010_2025.csv")

if _stocks_csv and _gold_csv and _returns_csv:
    print("Pre-saved data found in /kaggle/input — loading from disk...")
    stock_prices = pd.read_csv(_stocks_csv,  index_col=0, parse_dates=True).sort_index()
    gold         = pd.read_csv(_gold_csv,    index_col=0, parse_dates=True).sort_index()
    data         = pd.read_csv(_returns_csv, index_col=0, parse_dates=True).sort_index()
    final_stocks = stock_prices.columns.tolist()
    usd_inr = gold[["USD_INR"]].copy() if "USD_INR" in gold.columns else pd.DataFrame()
    _PRELOADED = True
    print(f"Loaded {len(final_stocks)} stocks | {len(data)} return observations.")
else:
    print("No pre-saved data found — downloading from Yahoo Finance and FRED...")

    stock_prices = download_stock_closes(STOCKS, START_DATE, END_DATE, batch_size=12)
    stock_prices = stock_prices.sort_index()

    missing_tickers = [t for t in STOCKS if t not in stock_prices.columns]
    for ticker in missing_tickers:
        try:
            single = download_single_close(ticker, START_DATE, END_DATE, label=ticker)
            if not single.empty:
                stock_prices[ticker] = single.iloc[:, 0].reindex(stock_prices.index)
        except Exception as e:
            print(f"Retry failed for {ticker}: {e}")

    stock_prices = stock_prices.loc[:, ~stock_prices.columns.duplicated()]
    stock_prices = stock_prices.dropna(axis=1, how="all").sort_index()
    final_stocks = stock_prices.columns.tolist()

    print("Final stock universe size:", len(final_stocks))
    if len(final_stocks) == 0:
        raise ValueError("No stock data downloaded. Check internet access or ticker availability.")

    print("Downloading gold futures (GC=F)...")
    gold_raw = download_single_close("GC=F", START_DATE, END_DATE, label="Gold_USD")
    if gold_raw.empty:
        raise ValueError("Gold futures download failed.")
    gold_usd = gold_raw.copy()
    if "Gold_USD" not in gold_usd.columns:
        gold_usd.columns = ["Gold_USD"]

    print("Downloading USD/INR from FRED...")
    fx_url  = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXINUS"
    usd_inr = pd.read_csv(fx_url)
    usd_inr.rename(columns={usd_inr.columns[0]: "DATE"}, inplace=True)
    usd_inr["DATE"]    = pd.to_datetime(usd_inr["DATE"])
    usd_inr["DEXINUS"] = usd_inr["DEXINUS"].replace(".", np.nan)
    usd_inr["USD_INR"] = pd.to_numeric(usd_inr["DEXINUS"], errors="coerce")
    usd_inr = usd_inr[["DATE", "USD_INR"]].dropna().set_index("DATE").sort_index()

    usd_inr_aligned = usd_inr.reindex(gold_usd.index).ffill().bfill()
    gold = pd.concat([gold_usd["Gold_USD"], usd_inr_aligned["USD_INR"]], axis=1).dropna()
    gold["Gold_10g_INR"] = gold["Gold_USD"] * gold["USD_INR"] * (10.0 / GRAMS_PER_OUNCE)
    print("Gold observations:", len(gold))

    stock_returns = np.log(stock_prices).diff()
    gold_returns  = np.log(gold["Gold_10g_INR"]).diff()
    gold_returns  = gold_returns.reindex(stock_returns.index)

    data = pd.concat([stock_returns, gold_returns.rename("Gold")], axis=1)
    data = data.replace([np.inf, -np.inf], np.nan).dropna()
    print("Final returns dataset shape:", data.shape)

    if len(data) <= 300:
        raise ValueError("Dataset too short for wavelet analysis.")

    stock_prices.to_csv(os.path.join(OUTPUT_DIR, "Stocks_Prices_2010_2025.csv"))
    gold.to_csv(os.path.join(OUTPUT_DIR, "Gold_Price_2010_2025.csv"))
    data.to_csv(os.path.join(OUTPUT_DIR, "Returns_Aligned_2010_2025.csv"))
    print("CSVs saved to", OUTPUT_DIR, "— add them as a Kaggle dataset for faster reruns.")

print("\nData ready. Shape:", data.shape)


In [ ]:

def modwt_mra_components(series, wavelet=WAVELET, level=LEVELS):
    x = np.asarray(series, dtype=float)
    n_orig = len(x)

    pad_len = int(2**np.ceil(np.log2(max(n_orig, 2**level))))
    x_pad  = np.pad(x, (0, pad_len - n_orig), mode="reflect")

    coeffs = pywt.swt(x_pad, wavelet, level=level, trim_approx=False)

    out = {}
    for detail_level in range(1, level + 1):
        cD = coeffs[detail_level - 1][1]   # detail at this level
        rec_coeffs = [(np.zeros_like(coeffs[k][0]),
                       coeffs[k][1] if k == detail_level - 1
                       else np.zeros_like(coeffs[k][1]))
                      for k in range(level)]
        rec = pywt.iswt(rec_coeffs, wavelet)[:n_orig]
        out[detail_level] = rec

    approx_coeffs = [(coeffs[k][0] if k == level - 1
                      else np.zeros_like(coeffs[k][0]),
                      np.zeros_like(coeffs[k][1]))
                     for k in range(level)]
    out["A"] = pywt.iswt(approx_coeffs, wavelet)[:n_orig]
    return out

wavelet_mra_components = modwt_mra_components
print("MODWT (non-decimating, Percival & Walden 2000) loaded — "
      "all scales retain full N observations.")


In [ ]:
print("Running wavelet decomposition...")

gold_wavelet   = wavelet_mra_components(data["Gold"].values)
stock_wavelets = {s: wavelet_mra_components(data[s].values) for s in final_stocks}

scale_data = {}
for scale in SCALES:
    df = pd.DataFrame({s: stock_wavelets[s][scale] for s in final_stocks})
    df["Gold"] = gold_wavelet[scale]
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    scale_data[scale] = df
    df.to_csv(os.path.join(OUTPUT_DIR, f"Wavelet_Scale{scale}.csv"), index=False)

print("Wavelet decomposition completed.")

In [ ]:
summary_rows = []
for scale in SCALES:
    df = scale_data[scale]
    for asset in df.columns:
        mean_v, std_v, sk_v, ku_v, jb_stat, jb_p = safe_stats(df[asset])
        summary_rows.append({
            "Scale": scale, "Asset": asset,
            "Mean": mean_v, "Std_Dev": std_v,
            "Skewness": sk_v, "Excess_Kurtosis": ku_v,
            "JB_Stat": jb_stat, "JB_p_value": jb_p
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "Distributional_Diagnostics.csv"), index=False)
print("Distributional diagnostics completed.")

fig, ax = plt.subplots(figsize=(8, 6))
stock_kurt = [
    summary_df[(summary_df["Scale"] == s) & (summary_df["Asset"] != "Gold")
               ]["Excess_Kurtosis"].dropna().values
    for s in SCALES
]
ax.boxplot(stock_kurt, labels=["Scale 1", "Scale 2", "Scale 3"])
for scale in SCALES:
    val = summary_df[(summary_df["Asset"] == "Gold") &
                     (summary_df["Scale"] == scale)]["Excess_Kurtosis"].values
    if len(val) and np.isfinite(val[0]):
        ax.scatter(scale, val[0], s=80, zorder=5, label=f"Gold S{scale}")
ax.set_ylabel("Excess Kurtosis")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "Kurtosis_Boxplot.png"), dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
stock_std = [
    summary_df[(summary_df["Scale"] == s) & (summary_df["Asset"] != "Gold")
               ]["Std_Dev"].dropna().values
    for s in SCALES
]
ax.boxplot(stock_std, labels=["Scale 1", "Scale 2", "Scale 3"])
for scale in SCALES:
    val = summary_df[(summary_df["Asset"] == "Gold") &
                     (summary_df["Scale"] == scale)]["Std_Dev"].values
    if len(val) and np.isfinite(val[0]):
        ax.scatter(scale, val[0], s=80, zorder=5, label=f"Gold S{scale}")
ax.set_ylabel("Standard Deviation")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "Volatility_Boxplot.png"), dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
gold["Gold_10g_INR"].plot(ax=ax)
ax.set_ylabel("INR per 10 grams")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "Gold_Price_INR.png"), dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(data["Gold"].values, alpha=0.4, label="Gold Returns")
ax.plot(gold_wavelet[1], label="Short-term (D1)")
ax.plot(gold_wavelet[2], label="Medium-term (D2)")
ax.plot(gold_wavelet[3], label="Long-term (D3)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "Gold_Wavelet_Components.png"), dpi=300)
plt.show()

example_stock = final_stocks[0]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(scale_data[3][example_stock].values, label=example_stock)
ax.plot(scale_data[3]["Gold"].values, label="Gold")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "LongHorizon_StockVsGold.png"), dpi=300)
plt.show()

In [ ]:
vol_share_rows = []
for stock in final_stocks:
    original_std = data[stock].std()
    for scale in SCALES:
        scale_std = scale_data[scale][stock].std()
        ratio = (scale_std / original_std
                 if original_std and np.isfinite(original_std) and original_std != 0
                 else np.nan)
        vol_share_rows.append({"Stock": stock, "Scale": scale, "Volatility_Share": ratio})

vol_share_df = pd.DataFrame(vol_share_rows)
vol_share_df.to_csv(os.path.join(OUTPUT_DIR, "Volatility_Share.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 6))
share_by_scale = [vol_share_df[vol_share_df["Scale"] == s]["Volatility_Share"].dropna().values
                  for s in SCALES]
ax.boxplot(share_by_scale, labels=["Scale 1", "Scale 2", "Scale 3"])
ax.set_ylabel("Volatility Share")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "VolatilityShare_Boxplot.png"), dpi=300)
plt.show()


In [ ]:

print("\nStarting parallelized Kernel QQ estimation for all stocks...\n")
print("This is computationally intensive. Progress is checkpointed to disk.")

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "kernel_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_ckpt_input = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith("_alpha.npy"):
            _ckpt_input = root
            break
    if _ckpt_input:
        break

if _ckpt_input and _ckpt_input != CHECKPOINT_DIR:
    import shutil
    for f in os.listdir(_ckpt_input):
        shutil.copy(os.path.join(_ckpt_input, f), CHECKPOINT_DIR)
    print("Checkpoints loaded from dataset.")
def _compute_bandwidth_and_quantiles(x_raw, quantiles):
    n         = len(x_raw)
    iqr       = np.subtract(*np.percentile(x_raw, [75, 25]))
    sigma     = np.std(x_raw, ddof=1)
    scale_bw  = min(sigma, iqr / 1.34) if iqr > 0 else sigma
    bandwidth = 0.9 * scale_bw * (n ** (-1 / 5))
    if (not np.isfinite(bandwidth)) or bandwidth <= 0:
        bandwidth = (sigma if sigma > 0 else 1.0) * (n ** (-1 / 5)) + 1e-8
    tau_q_arr = np.quantile(x_raw, quantiles)   # shape (nq,) — computed once
    return bandwidth, tau_q_arr

def _kernel_qq_one_asset(asset, scale, stock_arr, gold_arr, quantiles):
    import numpy as np
    import cvxpy as cp
    import warnings
    warnings.filterwarnings("ignore")

    def gaussian_kernel(u):
        return np.exp(-0.5 * u ** 2) / np.sqrt(2 * np.pi)

    def solve_wqr(x, y, weights, theta):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        w = np.asarray(weights, dtype=float)
        n = len(x)
        if n == 0:
            return np.nan, np.nan
        w = np.maximum(w, 1e-12)
        w = w / w.sum()
        alpha_v  = cp.Variable()
        beta_v   = cp.Variable()
        u_plus   = cp.Variable(n, nonneg=True)
        u_minus  = cp.Variable(n, nonneg=True)
        constraints = [y - alpha_v - beta_v * x == u_plus - u_minus]
        objective   = cp.Minimize(
            cp.sum(cp.multiply(w, theta * u_plus + (1 - theta) * u_minus))
        )
        problem = cp.Problem(objective, constraints)
        for solver in ["CLARABEL", "ECOS", "SCS"]:
            try:
                problem.solve(solver=solver, warm_start=True, verbose=False)
                if (beta_v.value is not None and np.isfinite(beta_v.value)
                        and alpha_v.value is not None):
                    return float(beta_v.value), float(alpha_v.value)
            except Exception:
                continue
        return np.nan, np.nan

    x_raw = np.asarray(stock_arr, dtype=float)
    y_raw = np.asarray(gold_arr,  dtype=float)
    n     = len(x_raw)
    nq    = len(quantiles)

    if n < 10:
        nan_mat = np.full((nq, nq), np.nan)
        return scale, asset, nan_mat, nan_mat

    bandwidth, tau_q_arr = _compute_bandwidth_and_quantiles(x_raw, quantiles)

    beta_matrix  = np.full((nq, nq), np.nan)
    alpha_matrix = np.full((nq, nq), np.nan)   # intercept stored for SE pass

    for j, tau in enumerate(quantiles):
        x_q     = tau_q_arr[j]                 
        x_dev   = x_raw - x_q
        weights = gaussian_kernel((x_raw - x_q) / bandwidth)
        weights = np.maximum(weights, 1e-12)
        weights = weights / weights.sum()
        for i, theta in enumerate(quantiles):
            b, a = solve_wqr(x_dev, y_raw, weights, theta)
            beta_matrix[i, j]  = b
            alpha_matrix[i, j] = a

    return scale, asset, beta_matrix, alpha_matrix

tasks_beta = []
for scale in SCALES:
    for asset in scale_data[scale].columns:
        if asset == "Gold":
            continue
        ckpt = os.path.join(CHECKPOINT_DIR, f"{scale}_{asset.replace('/', '_')}.npy")
        if not os.path.exists(ckpt):
            tasks_beta.append((
                asset, scale,
                scale_data[scale][asset].values,
                scale_data[scale]["Gold"].values,
                QUANTILES
            ))

total_tasks = sum(
    1 for scale in SCALES
    for asset in scale_data[scale].columns
    if asset != "Gold"
)
print(f"Beta tasks: {total_tasks}  |  Already checkpointed: {total_tasks - len(tasks_beta)}  |  Remaining: {len(tasks_beta)}")

if tasks_beta:
    new_beta_results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
        delayed(_kernel_qq_one_asset)(asset, scale, stock_arr, gold_arr, quantiles)
        for asset, scale, stock_arr, gold_arr, quantiles in tasks_beta
    )
    for scale, asset, beta_matrix, alpha_matrix in new_beta_results:
        ckpt      = os.path.join(CHECKPOINT_DIR, f"{scale}_{asset.replace('/', '_')}.npy")
        ckpt_alph = os.path.join(CHECKPOINT_DIR, f"{scale}_{asset.replace('/', '_')}_alpha.npy")
        np.save(ckpt,      beta_matrix)
        np.save(ckpt_alph, alpha_matrix)
    print(f"\nCheckpointed {len(new_beta_results)} new beta + alpha matrices.")

beta_store = {}
se_store   = {}
qq_rows    = []
crash_cols = [idx for idx, q in enumerate(QUANTILES) if q <= CRASH_CUTOFF]

for scale in SCALES:
    for asset in scale_data[scale].columns:
        if asset == "Gold":
            continue
        key       = f"{scale}_{asset.replace('/', '_')}"
        ckpt_beta = os.path.join(CHECKPOINT_DIR, f"{key}.npy")
        ckpt_pval = os.path.join(CHECKPOINT_DIR, f"{key}_pval.npy")
        ckpt_se   = os.path.join(CHECKPOINT_DIR, f"{key}_se.npy")

        if not os.path.exists(ckpt_beta):
            continue

        bm = np.load(ckpt_beta)
        beta_store[(scale, asset)] = bm

        sem = np.load(ckpt_se)   if os.path.exists(ckpt_se)   else np.full_like(bm, np.nan)
        se_store[(scale, asset)]   = sem

        crash_beta = np.nanmedian(bm[:, crash_cols])
        verdict = classify_crash_beta(crash_beta)

        qq_rows.append({
            "Scale"      : scale,
            "Asset"      : asset,
            "Crash_Beta" : crash_beta,
            "Verdict"    : verdict,
        })

qq_results = pd.DataFrame(qq_rows).sort_values(["Scale", "Asset"]).reset_index(drop=True)
qq_results.to_csv(os.path.join(OUTPUT_DIR, "QQ_Results_Kernel.csv"), index=False)

print("\n" + "=" * 40)
print("FULL QQ RESULTS (all assets, significance-gated)")
print("=" * 40)
print(qq_results.to_string(index=False))

print("\n" + "=" * 40)
print("SUMMARY ACROSS STOCKS")
print("=" * 40)
for scale in SCALES:
    subset = qq_results[qq_results["Scale"] == scale]
    print(f"\nScale {scale} (n={len(subset)})")
    print(f"  Safe Haven : {subset['Verdict'].str.startswith('SAFE HAVEN').sum()}")
    print(f"  Hedge      : {subset['Verdict'].str.startswith('HEDGE').sum()}")
    print(f"  Co-Movement: {subset['Verdict'].str.startswith('CO-MOVEMENT').sum()}")


In [ ]:

print("\nRunning STATIONARY BLOCK bootstrap (fast: beta_store reweighting)...")
print("B=500 replications | expected block length l=10 | no cvxpy re-solve")

B_BOOT      = 500
CRASH_Q_LOW = 0.05
CRASH_Q_HIGH= 0.10
BOOT_ALPHA  = 0.05
BLOCK_L     = 10

crash_idx = [i for i, q in enumerate(QUANTILES)
             if CRASH_Q_LOW <= q <= CRASH_Q_HIGH]

def _stationary_block_resample(n, l, rng):
    p    = 1.0 / l
    idxs = []
    while len(idxs) < n:
        start     = rng.integers(0, n)
        block_len = rng.geometric(p)
        for k in range(int(block_len)):
            idxs.append(int((start + k) % n))
            if len(idxs) == n:
                break
    return np.array(idxs[:n], dtype=int)

def _fast_bootstrap_one(asset, scale, stock_arr, gold_arr, quantiles,
                         crash_idx, beta_matrix,
                         B=500, alpha=0.05, block_l=10):
    import numpy as np

    rng = np.random.default_rng(42 + abs(hash(asset)) % 100000)

    def gaussian_kernel(u):
        return np.exp(-0.5 * u * u) / 2.5066282746310002  # 1/sqrt(2pi)

    def silverman_bw(x):
        n   = len(x)
        iqr = np.subtract(*np.percentile(x, [75, 25]))
        sig = np.std(x, ddof=1)
        sc  = min(sig, iqr / 1.34) if iqr > 0 else sig
        bw  = 0.9 * sc * (n ** (-1.0 / 5))
        return max(bw, 1e-8)

    x_raw = np.asarray(stock_arr, dtype=float)
    n     = len(x_raw)
    bw    = silverman_bw(x_raw)
    bm    = np.asarray(beta_matrix, dtype=float)   # 25 x 25 stored betas

    tau_qs = np.quantile(x_raw, quantiles)          # shape (nq,)

    def crash_beta_from_weights(x_data):
        betas = []
        for j in crash_idx:
            xq    = tau_qs[j]                         
            raw_w = gaussian_kernel((x_data - xq) / bw)
            raw_w = np.maximum(raw_w, 1e-12)
            w     = raw_w / raw_w.sum()                # normalised weights

            for i in crash_idx:
                b = bm[i, j]
                if np.isfinite(b):
                    betas.append(b * w.sum())          # w.sum()=1 on full data
        return np.nanmean(betas) if betas else np.nan

    obs_cells = [bm[i, j] for i in crash_idx for j in crash_idx
                 if np.isfinite(bm[i, j])]
    obs_beta  = np.nanmean(obs_cells) if obs_cells else np.nan

    boot_betas = []
    for _ in range(B):
        idx    = _stationary_block_resample(n, block_l, rng)
        x_boot = x_raw[idx]
        b_boot = crash_beta_from_weights(x_boot)
        if np.isfinite(b_boot):
            boot_betas.append(b_boot)

    if len(boot_betas) < 10:
        return scale, asset, obs_beta, np.nan, np.nan, np.nan

    boot_arr = np.array(boot_betas)
    ci_lo    = np.percentile(boot_arr, 100 * alpha / 2)
    ci_hi    = np.percentile(boot_arr, 100 * (1 - alpha / 2))
    p_val    = 2 * min(float(np.mean(boot_arr >= 0)),
                       float(np.mean(boot_arr <= 0)))

    return scale, asset, obs_beta, ci_lo, ci_hi, p_val

boot_tasks = []
for scale in SCALES:
    for asset in scale_data[scale].columns:
        if asset == "Gold":
            continue
        if (scale, asset) not in beta_store:
            print(f"  Warning: beta_store missing ({scale}, {asset}) — skipped")
            continue
        boot_tasks.append((
            asset, scale,
            scale_data[scale][asset].values,
            scale_data[scale]["Gold"].values,
            QUANTILES, crash_idx,
            beta_store[(scale, asset)]      
        ))

print(f"Bootstrap tasks: {len(boot_tasks)}  (using stored beta_store — no cvxpy)")
boot_results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(_fast_bootstrap_one)(
        asset, scale, sa, ga, q, ci, bm, B_BOOT, BOOT_ALPHA, BLOCK_L
    )
    for asset, scale, sa, ga, q, ci, bm in boot_tasks
)

boot_rows = []
for scale, asset, obs_b, ci_lo, ci_hi, pval in boot_results:
    ci_excludes_zero = (
        np.isfinite(ci_lo) and np.isfinite(ci_hi) and
        not (ci_lo <= 0 <= ci_hi)
    )
    sig = ci_excludes_zero and (np.isnan(pval) or pval < BOOT_ALPHA)
    if sig and obs_b < SAFE_HAVEN_THRESHOLD:
        verdict = "SAFE HAVEN"
    elif sig and obs_b < 0:
        verdict = "HEDGE"
    elif sig and obs_b >= 0:
        verdict = "CO-MOVEMENT"
    else:
        verdict = "NOT SIGNIFICANT"

    boot_rows.append({
        "Scale"           : scale,
        "Asset"           : asset,
        "Crash_Beta_Obs"  : obs_b,
        "CI_Lo_95"        : ci_lo,
        "CI_Hi_95"        : ci_hi,
        "Bootstrap_Pval"  : pval,
        "CI_Excludes_Zero": ci_excludes_zero,
        "Verdict"         : verdict,
        "Block_Length"    : BLOCK_L,
        "Method"          : "Stationary Block Bootstrap — fast reweighting (Politis-Romano 1994 / Ma-Kosorok 2005)"
    })

boot_df = pd.DataFrame(boot_rows).sort_values(["Scale", "Asset"]).reset_index(drop=True)
boot_df.to_csv(os.path.join(OUTPUT_DIR, "Bootstrap_Block_Results.csv"), index=False)

qq_results = boot_df.rename(columns={
    "Crash_Beta_Obs"  : "Crash_Beta",
    "Bootstrap_Pval"  : "Crash_Pval",
    "CI_Excludes_Zero": "Significant"
})

print("\n" + "="*55)
print("BOOTSTRAP (STATIONARY BLOCK l=10, FAST REWEIGHTING) RESULTS")
print("="*55)
print(boot_df.to_string(index=False))

print("\n── Scale Summary ──")
for scale in SCALES:
    sub = boot_df[boot_df["Scale"] == scale]
    print(f"  Scale {scale}: Safe Haven={(sub.Verdict=='SAFE HAVEN').sum()}  "
          f"Hedge={(sub.Verdict=='HEDGE').sum()}  "
          f"Co-Movement={(sub.Verdict=='CO-MOVEMENT').sum()}  "
          f"Not Sig={(sub.Verdict=='NOT SIGNIFICANT').sum()}")


In [ ]:
classification_df = qq_results.copy()
classification_df["Stock"]  = classification_df["Asset"]
classification_df["Sector"] = classification_df["Stock"].map(sector_map).fillna("Unknown")

classification_df["Classification"] = (
    classification_df["Verdict"]
    .str.replace(r" \(NS\)", "", regex=True)
    .map({
        "SAFE HAVEN"  : "SAFE_HAVEN",
        "HEDGE"       : "HEDGE",
        "CO-MOVEMENT" : "CO_MOVEMENT",
        "NA"          : "NA"
    })
)

pivot_table = classification_df.pivot(
    index=["Stock", "Sector"], columns="Scale", values="Classification"
)
pivot_table = pivot_table.reindex(columns=SCALES)
pivot_table.columns = [f"Scale{c}" for c in pivot_table.columns]

sector_summary_rows = []
for scale in SCALES:
    df_scale = classification_df[classification_df["Scale"] == scale]
    for sector in df_scale["Sector"].unique():
        df_sec   = df_scale[df_scale["Sector"] == sector]
        total    = len(df_sec)
        safe_cnt = len(df_sec[df_sec["Classification"] == "SAFE_HAVEN"])
        sector_summary_rows.append({
            "Sector": sector, "Scale": scale,
            "Total_Stocks": total, "Safe_Haven_Stocks": safe_cnt,
            "Safe_Haven_Probability": safe_cnt / total if total > 0 else np.nan
        })
sector_summary_df = pd.DataFrame(sector_summary_rows)

not_safe_df        = classification_df[classification_df["Classification"] != "SAFE_HAVEN"].copy()
sector_pattern_df  = (classification_df
                      .groupby(["Sector", "Scale", "Classification"])
                      .size()
                      .reset_index(name="Count"))

scale_summary_rows = []
for scale in SCALES:
    df_scale  = classification_df[classification_df["Scale"] == scale]
    total     = len(df_scale)
    safe_cnt  = len(df_scale[df_scale["Classification"] == "SAFE_HAVEN"])
    hedge_cnt = len(df_scale[df_scale["Classification"] == "HEDGE"])
    co_cnt    = len(df_scale[df_scale["Classification"] == "CO_MOVEMENT"])
    scale_summary_rows.append({
        "Scale": scale,
        "Safe_Haven": safe_cnt, "Hedge": hedge_cnt, "Co_Movement": co_cnt,
        "Total": total,

        "Safe_Haven_Probability": safe_cnt / total if total > 0 else np.nan
    })
scale_summary_df = pd.DataFrame(scale_summary_rows)

print("\nStock Classification Table (head)")
print(classification_df.head())
print("\nPivot Table (Stock vs Scales)")
print(pivot_table.head())
print("\nSector Safe Haven Summary")
print(sector_summary_df)
print("\nSector Patterns (head 20)")
print(sector_pattern_df.head(20))
print("\nScale Decay Summary")
print(scale_summary_df)

combined_xlsx = os.path.join(OUTPUT_DIR, "Gold_Stocks_Analysis_All.xlsx")
with pd.ExcelWriter(combined_xlsx, engine="openpyxl") as writer:
    summary_df.to_excel(writer,        sheet_name="Diagnostics",       index=False)
    vol_share_df.to_excel(writer,      sheet_name="Volatility_Share",  index=False)
    qq_results.to_excel(writer,        sheet_name="QQ_Results",        index=False)
    classification_df.to_excel(writer, sheet_name="Classification",    index=False)
    pivot_table.to_excel(writer,       sheet_name="Pivot_Table")
    sector_summary_df.to_excel(writer, sheet_name="Sector_Summary",    index=False)
    not_safe_df.to_excel(writer,       sheet_name="Non_SafeHaven",      index=False)
    sector_pattern_df.to_excel(writer, sheet_name="Sector_Patterns",   index=False)
    scale_summary_df.to_excel(writer,  sheet_name="Scale_Decay",       index=False)

classification_df.to_csv(os.path.join(OUTPUT_DIR, "Classification_Table.csv"),     index=False)
pivot_table.to_csv(      os.path.join(OUTPUT_DIR, "Classification_Pivot_Table.csv"))
sector_summary_df.to_csv(os.path.join(OUTPUT_DIR, "Sector_Summary.csv"),           index=False)
not_safe_df.to_csv(      os.path.join(OUTPUT_DIR, "Non_SafeHaven_Stocks.csv"),      index=False)
sector_pattern_df.to_csv(os.path.join(OUTPUT_DIR, "Sector_Patterns.csv"),          index=False)
scale_summary_df.to_csv( os.path.join(OUTPUT_DIR, "Scale_Decay.csv"),              index=False)

print("\nExcel file created:", combined_xlsx)


In [ ]:
individual_pdf = os.path.join(OUTPUT_DIR, "QQ_Individual_Heatmaps.pdf")
with PdfPages(individual_pdf) as pdf:
    for scale in SCALES:
        print(f"Saving individual heatmaps for Scale {scale} ...")
        for asset in final_stocks:
            if (scale, asset) not in beta_store:
                continue
            bm  = beta_store[(scale, asset)]
            fig = plot_heatmap(
                bm,
                title=f"Gold vs {asset} (Scale {scale})"
            )
            pdf.savefig(fig)
            plt.close(fig)
print("Individual heatmaps saved to:", individual_pdf)

summary_pdf = os.path.join(OUTPUT_DIR, "QQ_Summarised_Heatmaps.pdf")
with PdfPages(summary_pdf) as pdf:

    nifty_asset = "^NSEI"
    for scale in SCALES:
        if (scale, nifty_asset) not in beta_store:
            continue
        bm = beta_store[(scale, nifty_asset)]
        fig = plot_heatmap(
            bm,
            title=f"Gold vs NIFTY (Kernel QQ, Scale {scale})"
        )
        pdf.savefig(fig)
        plt.close(fig)

    for scale in SCALES:
        matrices = [beta_store[(scale, a)]
                    for a in final_stocks
                    if a != "^NSEI" and (scale, a) in beta_store]
        if not matrices:
            continue
        avg_beta = np.nanmean(matrices, axis=0)
        fig = plot_heatmap(
            avg_beta,
            title=f"Average Gold-Stock Dependence (Scale {scale})"
        )
        pdf.savefig(fig)
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(scale_summary_df["Scale"],
            scale_summary_df["Safe_Haven_Probability"],
            marker="o")
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["Short", "Medium", "Long"])
    ax.set_ylabel("Safe Haven Probability")
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

print("Summary heatmaps saved to:", summary_pdf)


In [ ]:
print("\nRunning PCC-based MST for each scale...")
mst_pcc = {}
for scale in SCALES:
    print(f"PCC MST — Scale {scale}")
    df   = scale_data[scale].drop(columns=["^NSEI"], errors="ignore").copy()
    corr = df.corr(method="pearson").clip(-0.999999, 0.999999)
    np.fill_diagonal(corr.values, 1.0)
    dist = pd.DataFrame(
        np.sqrt(np.maximum(0.0, 2 * (1 - corr))),
        index=corr.index, columns=corr.columns
    )
    pcc_path = os.path.join(OUTPUT_DIR, f"PCC_MST_Scale_{scale}.png")
    mst_obj = plot_mst_from_distance_matrix(
        dist, title=f"PCC-based MST (Scale {scale})",
        highlight_node="Gold", save_path=pcc_path,
        label_map=ticker_to_code
    )
    mst_pcc[scale] = {"dist": dist, "path": pcc_path, "mst": mst_obj}


In [ ]:
CHI_TILDA_Q = 0.10   # lower quantile threshold (upper = 1 - CHI_TILDA_Q)

print(f"\nRunning chi-tilda MST  (q_lower={CHI_TILDA_Q}, q_upper={1-CHI_TILDA_Q})...")


def compute_chi_tilda_matrix(df, q=CHI_TILDA_Q, tail="lower"):
    """
    Symmetric normalised chi-tilda matrix.

    Parameters
    ----------
    df   : DataFrame of wavelet-scale returns (assets as columns)
    q    : quantile threshold (e.g. 0.10 for lower, 0.90 for upper)
    tail : "lower" → both below Q(q)
           "upper" → both above Q(q)

    Returns
    -------
    chi_df : symmetric DataFrame, values in [-1, 1]
    """
    assets = list(df.columns)
    na     = len(assets)
    mat    = np.zeros((na, na))

    for ii, i in enumerate(assets):
        Xi = df[i].values
        qi = np.nanquantile(Xi, q)
        for jj in range(ii, na):
            j  = assets[jj]
            Xj = df[j].values
            qj = np.nanquantile(Xj, q)

            if tail == "lower":
                indicator = ((Xi <= qi) & (Xj <= qj)).astype(float)
                p = q
            else:
                indicator = ((Xi >= qi) & (Xj >= qj)).astype(float)
                p = 1.0 - q

            chi_raw   = float(np.nanmean(indicator))
            denom     = p * (1.0 - p)
            if denom == 0 or not np.isfinite(denom):
                chi_t = 0.0
            else:
                chi_t = float(np.clip((chi_raw - p ** 2) / denom, -1.0, 1.0))
            if not np.isfinite(chi_t):
                chi_t = 0.0

            mat[ii, jj] = chi_t
            mat[jj, ii] = chi_t

    np.fill_diagonal(mat, 1.0)
    return pd.DataFrame(mat, index=assets, columns=assets)


def chi_tilda_mst(df, scale, q=CHI_TILDA_Q, tail="lower", save_path=None):
    """Build distance matrix and plot MST from chi-tilda matrix."""
    label = f"{'Lower' if tail=='lower' else 'Upper'} Tail"
    thresh_str = f"q={q}" if tail == "lower" else f"q={q} (upper {int(round(q*100))}th pct)"
    print(f"  Chi-tilda {label} MST — Scale {scale}  [{thresh_str}]")

    chi_mat = compute_chi_tilda_matrix(df, q=q, tail=tail)
    chi_mat = chi_mat.fillna(0.0).clip(-0.999999, 1.0)

    dist = pd.DataFrame(
        np.sqrt(np.maximum(0.0, 2.0 * (1.0 - chi_mat.values))),
        index=chi_mat.index, columns=chi_mat.columns
    )

    plot_mst_from_distance_matrix(
        dist,
        title=(f"Chi-tilda {label} MST  (normalised joint-quantile, {thresh_str})"
               f" — Scale {scale}"),
        highlight_node="Gold",
        save_path=save_path,
        label_map=ticker_to_code
    )
    return chi_mat, dist


mst_chitilda_lower = {}
mst_chitilda_upper = {}

for scale in SCALES:
    df_s = scale_data[scale].drop(columns=["^NSEI"], errors="ignore").copy()

    chi_lo, dist_lo = chi_tilda_mst(
        df_s, scale, q=CHI_TILDA_Q, tail="lower",
        save_path=os.path.join(OUTPUT_DIR,
                               f"ChiTilda_LowerTail_MST_Scale_{scale}.png")
    )
    chi_up, dist_up = chi_tilda_mst(
        df_s, scale, q=1.0 - CHI_TILDA_Q, tail="upper",
        save_path=os.path.join(OUTPUT_DIR,
                               f"ChiTilda_UpperTail_MST_Scale_{scale}.png")
    )

    mst_chitilda_lower[scale] = {"chi": chi_lo, "dist": dist_lo}
    mst_chitilda_upper[scale] = {"chi": chi_up, "dist": dist_up}

    print(f"\n  Scale {scale} — Chi-tilda (lower {int(CHI_TILDA_Q*100)}th pct)  Gold row:")
    print(chi_lo.loc["Gold"].drop("Gold").sort_values(ascending=False).round(4).to_string())

    print(f"\n  Scale {scale} — Chi-tilda (upper {int((1-CHI_TILDA_Q)*100)}th pct)  Gold row:")
    print(chi_up.loc["Gold"].drop("Gold").sort_values(ascending=False).round(4).to_string())

print("\nChi-tilda MSTs complete.")
print(f"  Stored in: mst_chitilda_lower, mst_chitilda_upper")


In [ ]:
print("\nExtracting chi-tilda MST network metrics for Gold...")

chitilda_metrics_path = os.path.join(OUTPUT_DIR, "ChiTilda_MST_Network_Metrics.xlsx")


def dist_to_mst_ct(dist_df):
    """Build NetworkX MST from a distance DataFrame."""
    G = nx.Graph()
    nodes = list(dist_df.columns)
    for i in nodes:
        for j in nodes:
            if i < j:
                G.add_edge(i, j, weight=float(dist_df.loc[i, j]))
    return nx.minimum_spanning_tree(G, weight="weight")


def gold_metrics_ct(mst, scale, mst_type, gold="Gold"):
    """Extract Gold-centric metrics from an MST graph."""
    deg   = dict(mst.degree())
    bc    = nx.betweenness_centrality(mst, weight="weight", normalized=True)
    cc    = nx.closeness_centrality(mst, distance="weight")
    hub   = max(deg, key=deg.get)

    nbrs  = list(mst.neighbors(gold)) if gold in mst else []
    ewts  = [mst[gold][nb]["weight"] for nb in nbrs] if nbrs else []

    try:
        hops  = nx.shortest_path_length(mst, gold, hub)
    except Exception:
        hops  = np.nan
    try:
        wdist = nx.shortest_path_length(mst, gold, hub, weight="weight")
    except Exception:
        wdist = np.nan
    try:
        hop_all = nx.single_source_shortest_path_length(mst, gold)
        hop_all.pop(gold, None)
        mean_hops = np.mean(list(hop_all.values()))
    except Exception:
        mean_hops = np.nan

    nbr_sectors = ", ".join(sorted(set(sector_map.get(nb, "Unknown") for nb in nbrs)))

    return {
        "Scale"                 : scale,
        "MST_Type"              : mst_type,
        "Gold_Degree"           : deg.get(gold, np.nan),
        "Gold_Is_Leaf"          : int(deg.get(gold, 0) == 1),
        "Gold_Neighbours"       : ", ".join(sorted(nbrs)),
        "Gold_Neighbour_Sectors": nbr_sectors,
        "Gold_Edge_Wt_Sum"      : np.sum(ewts)  if ewts else np.nan,
        "Gold_Edge_Wt_Mean"     : np.mean(ewts) if ewts else np.nan,
        "Hub_Node"              : hub,
        "Hub_Degree"            : deg[hub],
        "Gold_Hops_to_Hub"      : hops,
        "Gold_WtDist_to_Hub"    : wdist,
        "Gold_Mean_Hops_All"    : mean_hops,
        "Gold_Betweenness"      : bc.get(gold, np.nan),
        "Gold_Closeness"        : cc.get(gold, np.nan),
    }


def centrality_table_ct(mst, label):
    """Full centrality table for every node in one MST."""
    deg = dict(mst.degree())
    bc  = nx.betweenness_centrality(mst, weight="weight", normalized=True)
    cc  = nx.closeness_centrality(mst, distance="weight")
    rows = []
    for node in mst.nodes():
        rows.append({
            "MST"        : label,
            "Node"       : node,
            "Degree"     : deg[node],
            "Betweenness": round(bc[node], 6),
            "Closeness"  : round(cc[node], 6),
            "Is_Gold"    : int(node == "Gold"),
        })
    return pd.DataFrame(rows).sort_values("Betweenness", ascending=False).reset_index(drop=True)


ct_lower_msts = {}
ct_upper_msts = {}

for scale in SCALES:
    ct_lower_msts[scale] = dist_to_mst_ct(mst_chitilda_lower[scale]["dist"])
    ct_upper_msts[scale] = dist_to_mst_ct(mst_chitilda_upper[scale]["dist"])

ct_all_rows = []
for scale in SCALES:
    ct_all_rows.append(gold_metrics_ct(ct_lower_msts[scale], scale, "ChiTilda_Lower"))
    ct_all_rows.append(gold_metrics_ct(ct_upper_msts[scale], scale, "ChiTilda_Upper"))

df_ct_all = pd.DataFrame(ct_all_rows)

df_ct_lower = (df_ct_all[df_ct_all["MST_Type"] == "ChiTilda_Lower"]
               .set_index("Scale").drop(columns="MST_Type"))
df_ct_upper = (df_ct_all[df_ct_all["MST_Type"] == "ChiTilda_Upper"]
               .set_index("Scale").drop(columns="MST_Type"))

ct_asym_rows = []
for scale in SCALES:
    lo = df_ct_lower.loc[scale]
    up = df_ct_upper.loc[scale]
    ct_asym_rows.append({
        "Scale"                         : scale,
        "Lower_Gold_Degree"             : lo["Gold_Degree"],
        "Upper_Gold_Degree"             : up["Gold_Degree"],
        "Degree_Asym_Lower_minus_Upper" : lo["Gold_Degree"] - up["Gold_Degree"],
        "Lower_Gold_Hops_to_Hub"        : lo["Gold_Hops_to_Hub"],
        "Upper_Gold_Hops_to_Hub"        : up["Gold_Hops_to_Hub"],
        "Lower_Gold_Betweenness"        : lo["Gold_Betweenness"],
        "Upper_Gold_Betweenness"        : up["Gold_Betweenness"],
        "Lower_Gold_Neighbours"         : lo["Gold_Neighbours"],
        "Upper_Gold_Neighbours"         : up["Gold_Neighbours"],
        "Lower_Gold_Is_Leaf"            : lo["Gold_Is_Leaf"],
        "Upper_Gold_Is_Leaf"            : up["Gold_Is_Leaf"],
    })
df_ct_asym = pd.DataFrame(ct_asym_rows).set_index("Scale")

horizon = {1: "Short", 2: "Medium", 3: "Long"}
ct_evo_rows = []
for scale in SCALES:
    ct_evo_rows.append({
        "Scale"                    : scale,
        "Horizon"                  : horizon[scale],
        "Lower_Gold_Degree"        : df_ct_lower.loc[scale, "Gold_Degree"],
        "Lower_Gold_Is_Leaf"       : df_ct_lower.loc[scale, "Gold_Is_Leaf"],
        "Lower_Gold_Hops_Hub"      : df_ct_lower.loc[scale, "Gold_Hops_to_Hub"],
        "Lower_Gold_Neighbours"    : df_ct_lower.loc[scale, "Gold_Neighbours"],
        "Lower_Gold_Betweenness"   : df_ct_lower.loc[scale, "Gold_Betweenness"],
        "Upper_Gold_Degree"        : df_ct_upper.loc[scale, "Gold_Degree"],
        "Upper_Gold_Is_Leaf"       : df_ct_upper.loc[scale, "Gold_Is_Leaf"],
        "Upper_Gold_Hops_Hub"      : df_ct_upper.loc[scale, "Gold_Hops_to_Hub"],
        "Upper_Gold_Neighbours"    : df_ct_upper.loc[scale, "Gold_Neighbours"],
        "Upper_Gold_Betweenness"   : df_ct_upper.loc[scale, "Gold_Betweenness"],
    })
df_ct_evo = pd.DataFrame(ct_evo_rows).set_index("Scale")

ct_cent_frames = []
for scale in SCALES:
    ct_cent_frames.append(centrality_table_ct(ct_lower_msts[scale],
                                               f"ChiTilda_Lower_Scale{scale}"))
    ct_cent_frames.append(centrality_table_ct(ct_upper_msts[scale],
                                               f"ChiTilda_Upper_Scale{scale}"))
df_ct_cent = pd.concat(ct_cent_frames, ignore_index=True)

with pd.ExcelWriter(chitilda_metrics_path, engine="openpyxl") as writer:
    df_ct_all.to_excel(writer,    sheet_name="All_Gold_Metrics",      index=False)
    df_ct_lower.to_excel(writer,  sheet_name="Lower_Gold_Metrics")
    df_ct_upper.to_excel(writer,  sheet_name="Upper_Gold_Metrics")
    df_ct_asym.to_excel(writer,   sheet_name="Tail_Asymmetry")
    df_ct_evo.to_excel(writer,    sheet_name="Scale_Evolution")
    df_ct_cent.to_excel(writer,   sheet_name="All_Node_Centrality",   index=False)

print(f"Chi-tilda MST metrics saved → {chitilda_metrics_path}")

print("\n" + "="*60)
print("GOLD CHI-TILDA MST METRICS — SUMMARY")
print("="*60)

for scale in SCALES:
    print(f"\n── Scale {scale} ({horizon[scale]}) ──")
    for mtype, df_m in [("ChiTilda Lower", df_ct_lower), ("ChiTilda Upper", df_ct_upper)]:
        r        = df_m.loc[scale]
        leaf_tag = " [LEAF]" if r["Gold_Is_Leaf"] else ""
        print(f"  {mtype:18s}  degree={int(r['Gold_Degree'])}  "
              f"hops_to_hub={r['Gold_Hops_to_Hub']}  "
              f"betweenness={r['Gold_Betweenness']:.4f}"
              f"{leaf_tag}  → {r['Gold_Neighbours']}")

print("\n── Tail Asymmetry (Lower degree − Upper degree) ──")
print(df_ct_asym[["Lower_Gold_Degree", "Upper_Gold_Degree",
                   "Degree_Asym_Lower_minus_Upper"]].to_string())


mst metric calculations

In [ ]:
print("\nExtracting MST network metrics for Gold...")

mst_metrics_path = os.path.join(OUTPUT_DIR, "MST_Network_Metrics.xlsx")

def dist_to_mst(dist_df):
    G = nx.Graph()
    nodes = list(dist_df.columns)
    for i in nodes:
        for j in nodes:
            if i < j:
                G.add_edge(i, j, weight=float(dist_df.loc[i, j]))
    return nx.minimum_spanning_tree(G, weight="weight")

def gold_metrics(mst, scale, mst_type, gold="Gold"):
    deg   = dict(mst.degree())
    bc    = nx.betweenness_centrality(mst, weight="weight", normalized=True)
    cc    = nx.closeness_centrality(mst, distance="weight")
    hub   = max(deg, key=deg.get)

    nbrs  = list(mst.neighbors(gold)) if gold in mst else []
    ewts  = [mst[gold][nb]["weight"] for nb in nbrs]

    try:
        hops  = nx.shortest_path_length(mst, gold, hub)
    except Exception:
        hops  = np.nan
    try:
        wdist = nx.shortest_path_length(mst, gold, hub, weight="weight")
    except Exception:
        wdist = np.nan
    try:
        hop_all = nx.single_source_shortest_path_length(mst, gold)
        hop_all.pop(gold, None)
        mean_hops = np.mean(list(hop_all.values()))
    except Exception:
        mean_hops = np.nan

    nbr_sectors = ", ".join(sorted(set(sector_map.get(nb, "Unknown") for nb in nbrs)))

    return {
        "Scale"                 : scale,
        "MST_Type"              : mst_type,
        "Gold_Degree"           : deg.get(gold, np.nan),
        "Gold_Is_Leaf"          : int(deg.get(gold, 0) == 1),
        "Gold_Neighbours"       : ", ".join(sorted(nbrs)),
        "Gold_Neighbour_Sectors": nbr_sectors,
        "Gold_Edge_Wt_Sum"      : np.sum(ewts)  if ewts else np.nan,
        "Gold_Edge_Wt_Mean"     : np.mean(ewts) if ewts else np.nan,
        "Hub_Node"              : hub,
        "Hub_Degree"            : deg[hub],
        "Gold_Hops_to_Hub"      : hops,
        "Gold_WtDist_to_Hub"    : wdist,
        "Gold_Mean_Hops_All"    : mean_hops,
        "Gold_Betweenness"      : bc.get(gold, np.nan),
        "Gold_Closeness"        : cc.get(gold, np.nan),
    }

def centrality_table(mst, label):
    deg = dict(mst.degree())
    bc  = nx.betweenness_centrality(mst, weight="weight", normalized=True)
    cc  = nx.closeness_centrality(mst, distance="weight")
    rows = []
    for node in mst.nodes():
        rows.append({
            "MST"        : label,
            "Node"       : node,
            "Degree"     : deg[node],
            "Betweenness": round(bc[node], 6),
            "Closeness"  : round(cc[node], 6),
            "Is_Gold"    : int(node == "Gold"),
        })
    return pd.DataFrame(rows).sort_values("Betweenness", ascending=False).reset_index(drop=True)

pcc_msts   = {}
lower_msts = {}
upper_msts = {}

for scale in SCALES:
    df_s  = scale_data[scale].drop(columns=["^NSEI"], errors="ignore").copy()
    corr  = df_s.corr().clip(-0.999999, 0.999999)
    np.fill_diagonal(corr.values, 1.0)
    dist_pcc = pd.DataFrame(
        np.sqrt(np.maximum(0.0, 2 * (1 - corr))),
        index=corr.index, columns=corr.columns
    )
    pcc_msts[scale] = dist_to_mst(dist_pcc)

    lower_msts[scale] = dist_to_mst(mst_chitilda_lower[scale]["dist"])
    upper_msts[scale] = dist_to_mst(mst_chitilda_upper[scale]["dist"])

all_rows = []
for scale in SCALES:
    all_rows.append(gold_metrics(pcc_msts[scale],   scale, "PCC"))
    all_rows.append(gold_metrics(lower_msts[scale], scale, "Lower_Tail"))
    all_rows.append(gold_metrics(upper_msts[scale], scale, "Upper_Tail"))

df_all = pd.DataFrame(all_rows)

df_pcc   = df_all[df_all["MST_Type"] == "PCC"].set_index("Scale").drop(columns="MST_Type")
df_lower = df_all[df_all["MST_Type"] == "Lower_Tail"].set_index("Scale").drop(columns="MST_Type")
df_upper = df_all[df_all["MST_Type"] == "Upper_Tail"].set_index("Scale").drop(columns="MST_Type")

asym_rows = []
for scale in SCALES:
    lo = df_lower.loc[scale]
    up = df_upper.loc[scale]
    asym_rows.append({
        "Scale"                         : scale,
        "Lower_Gold_Degree"             : lo["Gold_Degree"],
        "Upper_Gold_Degree"             : up["Gold_Degree"],
        "Degree_Asym_Lower_minus_Upper" : lo["Gold_Degree"] - up["Gold_Degree"],
        "Lower_Gold_Hops_to_Hub"        : lo["Gold_Hops_to_Hub"],
        "Upper_Gold_Hops_to_Hub"        : up["Gold_Hops_to_Hub"],
        "Lower_Gold_Betweenness"        : lo["Gold_Betweenness"],
        "Upper_Gold_Betweenness"        : up["Gold_Betweenness"],
        "Lower_Gold_Neighbours"         : lo["Gold_Neighbours"],
        "Upper_Gold_Neighbours"         : up["Gold_Neighbours"],
        "Lower_Gold_Is_Leaf"            : lo["Gold_Is_Leaf"],
        "Upper_Gold_Is_Leaf"            : up["Gold_Is_Leaf"],
    })
df_asym = pd.DataFrame(asym_rows).set_index("Scale")

horizon = {1: "Short", 2: "Medium", 3: "Long"}
evo_rows = []
for scale in SCALES:
    evo_rows.append({
        "Scale"                : scale,
        "Horizon"              : horizon[scale],
        "PCC_Gold_Degree"      : df_pcc.loc[scale,   "Gold_Degree"],
        "PCC_Gold_Hops_Hub"    : df_pcc.loc[scale,   "Gold_Hops_to_Hub"],
        "PCC_Betweenness"      : df_pcc.loc[scale,   "Gold_Betweenness"],
        "PCC_Hub_Node"         : df_pcc.loc[scale,   "Hub_Node"],
        "PCC_Gold_Neighbours"  : df_pcc.loc[scale,   "Gold_Neighbours"],
        "Lower_Gold_Degree"    : df_lower.loc[scale, "Gold_Degree"],
        "Lower_Gold_Is_Leaf"   : df_lower.loc[scale, "Gold_Is_Leaf"],
        "Lower_Gold_Hops_Hub"  : df_lower.loc[scale, "Gold_Hops_to_Hub"],
        "Lower_Gold_Neighbours": df_lower.loc[scale, "Gold_Neighbours"],
        "Upper_Gold_Degree"    : df_upper.loc[scale, "Gold_Degree"],
        "Upper_Gold_Is_Leaf"   : df_upper.loc[scale, "Gold_Is_Leaf"],
        "Upper_Gold_Hops_Hub"  : df_upper.loc[scale, "Gold_Hops_to_Hub"],
        "Upper_Gold_Neighbours": df_upper.loc[scale, "Gold_Neighbours"],
    })
df_evo = pd.DataFrame(evo_rows).set_index("Scale")

cent_frames = []
for scale in SCALES:
    cent_frames.append(centrality_table(pcc_msts[scale],   f"PCC_Scale{scale}"))
    cent_frames.append(centrality_table(lower_msts[scale], f"Lower_Scale{scale}"))
    cent_frames.append(centrality_table(upper_msts[scale], f"Upper_Scale{scale}"))
df_cent = pd.concat(cent_frames, ignore_index=True)

with pd.ExcelWriter(mst_metrics_path, engine="openpyxl") as writer:
    df_all.to_excel(writer,   sheet_name="All_Gold_Metrics",      index=False)
    df_pcc.to_excel(writer,   sheet_name="PCC_Gold_Metrics")
    df_lower.to_excel(writer, sheet_name="LowerTail_Gold_Metrics")
    df_upper.to_excel(writer, sheet_name="UpperTail_Gold_Metrics")
    df_asym.to_excel(writer,  sheet_name="Tail_Asymmetry")
    df_evo.to_excel(writer,   sheet_name="Scale_Evolution")
    df_cent.to_excel(writer,  sheet_name="All_Node_Centrality",   index=False)

print(f"MST metrics saved → {mst_metrics_path}")

print("\n" + "="*60)
print("GOLD NETWORK METRICS — SUMMARY")
print("="*60)

for scale in SCALES:
    print(f"\n── Scale {scale} ({horizon[scale]}) ──")
    for mtype, df_m in [("PCC", df_pcc), ("Lower Tail", df_lower), ("Upper Tail", df_upper)]:
        r        = df_m.loc[scale]
        leaf_tag = " [LEAF]" if r["Gold_Is_Leaf"] else ""
        print(f"  {mtype:12s}  degree={int(r['Gold_Degree'])}  "
              f"hops_to_hub={r['Gold_Hops_to_Hub']}  "
              f"betweenness={r['Gold_Betweenness']:.4f}"
              f"{leaf_tag}  → {r['Gold_Neighbours']}")

print("\n── Tail Asymmetry (Lower degree − Upper degree) ──")
print(df_asym[["Lower_Gold_Degree", "Upper_Gold_Degree",
               "Degree_Asym_Lower_minus_Upper"]].to_string())

In [ ]:
print("\nGold vs USD/INR correlation check...")
usd_inr_ret   = np.log(usd_inr["USD_INR"]).diff()
gold_inr_ret  = np.log(gold["Gold_10g_INR"]).diff()
corr_df       = pd.concat(
    [gold_inr_ret.rename("Gold_INR"), usd_inr_ret.rename("USD_INR")],
    axis=1
).dropna()
corr_value = corr_df["Gold_INR"].corr(corr_df["USD_INR"])
print("Correlation between Gold (INR) and USD/INR returns:", corr_value)

fig, ax = plt.subplots(figsize=(10, 4))
corr_df["Gold_INR"].rolling(252).corr(corr_df["USD_INR"]).plot(ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "Rolling_Correlation_Gold_USDINR.png"), dpi=300)
plt.show()

In [ ]:
print("\nAdding all-stock Kernel QQ heatmaps to the individual PDF...")

with PdfPages(individual_pdf, "a") as pdf:
    for scale in SCALES:
        for asset in scale_data[scale].columns:
            if asset == "Gold":
                continue
            if (scale, asset) not in beta_store:
                print(f"  Skipping {asset} Scale {scale} — no beta matrix found.")
                continue
            bm         = beta_store[(scale, asset)]
            safe_label = asset.replace("^", "").replace(".NS", "")
            fig = plot_heatmap(
                bm,
                title=f"Gold vs {safe_label} (Kernel QQ, Scale {scale})"
            )
            pdf.savefig(fig)
            plt.close(fig)

print("All-stock Kernel QQ heatmaps clubbed into:", individual_pdf)

kernel_results_df = qq_results.copy()
kernel_results_df.rename(columns={
    "Crash_Beta": "Crash_Beta_Kernel",
    "Verdict":    "Verdict_Kernel",
}, inplace=True)

with pd.ExcelWriter(combined_xlsx, engine="openpyxl",
                    mode="a", if_sheet_exists="replace") as writer:
    kernel_results_df.to_excel(writer, sheet_name="Kernel_All_Stocks", index=False)

print("Kernel results added to Excel.")
print("\nKernel QQ summary:")
print(qq_results.to_string(index=False))


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

KEY_STOCKS = [
    "^NSEI",          # Index
    "HDFCBANK.NS",    # Banking
    "TCS.NS",         # IT
    "RELIANCE.NS",    # Energy
    "SUNPHARMA.NS",   # Pharma
    "TATASTEEL.NS",   # Metals
]

def plot_qq_3d_surface(beta_matrix, quantiles, asset, scale, ax=None, save_path=None):
    bm = np.array(beta_matrix, dtype=float)
    T, Q = np.meshgrid(quantiles, quantiles)   # T = tau grid, Q = theta grid

    fig = plt.figure(figsize=(8, 5))
    ax3d = fig.add_subplot(111, projection='3d')

    surf = ax3d.plot_surface(
        T, Q, bm,
        cmap='coolwarm', alpha=0.88,
        linewidth=0, antialiased=True
    )

    ax3d.plot_surface(
        T, Q, np.zeros_like(bm),
        alpha=0.12, color='gray'
    )

    safe_label = asset.replace('^', '').replace('.NS', '')
    scale_name = {1: 'Short (D1)', 2: 'Medium (D2)', 3: 'Long (D3)'}[scale]

    ax3d.set_xlabel('Stock Quantile (τ)', labelpad=8, fontsize=8)
    ax3d.set_ylabel('Gold Quantile (θ)', labelpad=8, fontsize=8)
    ax3d.set_zlabel('β₁(θ,τ)', labelpad=6, fontsize=8)
    ax3d.view_init(elev=28, azim=-55)   # camera angle: tilted enough to see tail corners
    ax3d.tick_params(axis='both', labelsize=6)

    fig.colorbar(surf, ax=ax3d, shrink=0.45, aspect=10, label='β coefficient')
    fig.tight_layout()
    return fig

print("Generating 3D surface plots...")

surface_pdf = os.path.join(OUTPUT_DIR, "QQ_3D_Surfaces.pdf")
with PdfPages(surface_pdf) as pdf:

    for scale in SCALES:
        matrices = [
            beta_store[(scale, a)]
            for a in final_stocks
            if a != '^NSEI' and (scale, a) in beta_store
        ]
        if not matrices:
            continue
        avg_bm = np.nanmean(matrices, axis=0)
        fig = plot_qq_3d_surface(avg_bm, QUANTILES, 'Average (All Stocks)', scale)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        print(f"  [3D] Average surface — Scale {scale}")

    for scale in SCALES:
        if (scale, '^NSEI') not in beta_store:
            continue
        fig = plot_qq_3d_surface(beta_store[(scale, '^NSEI')], QUANTILES, '^NSEI', scale)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        print(f"  [3D] NIFTY — Scale {scale}")

    for asset in [s for s in KEY_STOCKS if s != '^NSEI']:
        for scale in SCALES:
            if (scale, asset) not in beta_store:
                continue
            fig = plot_qq_3d_surface(beta_store[(scale, asset)], QUANTILES, asset, scale)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
        print(f"  [3D] {asset} — all scales")

    sectors_to_plot = ['Banking', 'IT', 'Pharma', 'Metals', 'FMCG']
    for sector in sectors_to_plot:
        sector_assets = [a for a in final_stocks if sector_map.get(a) == sector]
        matrices = [
            beta_store[(1, a)]
            for a in sector_assets
            if (1, a) in beta_store
        ]
        if not matrices:
            continue
        avg_bm = np.nanmean(matrices, axis=0)
        fig = plot_qq_3d_surface(avg_bm, QUANTILES, f'{sector} Avg', 1)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        print(f"  [3D] Sector avg — {sector} — Scale 1")

print(f"3D surface PDF saved to: {surface_pdf}")


In [ ]:
from statsmodels.regression.quantile_regression import QuantReg
import statsmodels.api as sm

def qr_vs_qqr_plot(asset, scale, beta_store, scale_data, quantiles,
                   save_path=None):
    if (scale, asset) not in beta_store:
        return None

    bm          = np.array(beta_store[(scale, asset)], dtype=float)
    qqr_avg     = np.nanmean(bm, axis=1)   # average over tau -> shape (len(quantiles),)

    df_scale    = scale_data[scale]
    if asset not in df_scale.columns or 'Gold' not in df_scale.columns:
        return None

    stock_arr   = df_scale[asset].values.astype(float)
    gold_arr    = df_scale['Gold'].values.astype(float)
    mask        = np.isfinite(stock_arr) & np.isfinite(gold_arr)
    stock_arr   = stock_arr[mask]
    gold_arr    = gold_arr[mask]

    if len(stock_arr) < 30:
        return None

    X           = sm.add_constant(stock_arr)
    qr_model    = QuantReg(gold_arr, X)
    qr_betas    = []
    for q in quantiles:
        try:
            res = qr_model.fit(q=float(q), max_iter=2000, p_tol=1e-6)
            qr_betas.append(res.params[1])
        except Exception:
            qr_betas.append(np.nan)
    qr_betas = np.array(qr_betas)

    safe_label  = asset.replace('^', '').replace('.NS', '')
    scale_name  = {1: 'Short (D1)', 2: 'Medium (D2)', 3: 'Long (D3)'}[scale]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(quantiles, qqr_avg,  'r-o', markersize=4, linewidth=1.4,
            label='QQR (averaged over τ)')
    ax.plot(quantiles, qr_betas, 'b-s', markersize=4, linewidth=1.4,
            label='Standard QR')
    ax.axhline(0, linestyle='--', color='gray', linewidth=0.8)
    ax.fill_between(quantiles, qqr_avg, qr_betas,
                    alpha=0.12, color='purple',
                    label='Divergence (QQR − QR)')
    ax.set_xlabel('Gold Return Quantile (θ)', fontsize=9)
    ax.set_ylabel('β coefficient', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300)
    return fig

print("Running QR vs QQR robustness checks...")

robustness_pdf = os.path.join(OUTPUT_DIR, "QR_vs_QQR_Robustness.pdf")
robustness_rows = []

with PdfPages(robustness_pdf) as pdf:

    for scale in SCALES:
        all_assets = [a for a in final_stocks if a != '^NSEI']

        matrices = [
            np.array(beta_store[(scale, a)], dtype=float)
            for a in all_assets if (scale, a) in beta_store
        ]
        if not matrices:
            continue
        avg_bm      = np.nanmean(matrices, axis=0)
        qqr_avg     = np.nanmean(avg_bm, axis=1)

        df_scale    = scale_data[scale]
        pool_x      = np.concatenate([df_scale[a].values for a in all_assets if a in df_scale.columns])
        pool_y      = np.tile(df_scale['Gold'].values, len([a for a in all_assets if a in df_scale.columns]))
        mask        = np.isfinite(pool_x) & np.isfinite(pool_y)
        pool_x      = pool_x[mask]; pool_y = pool_y[mask]

        X           = sm.add_constant(pool_x)
        qr_model    = QuantReg(pool_y, X)
        qr_betas    = []
        for q in QUANTILES:
            try:
                res = qr_model.fit(q=float(q), max_iter=2000)
                qr_betas.append(res.params[1])
            except Exception:
                qr_betas.append(np.nan)
        qr_betas = np.array(qr_betas)

        scale_name = {1: 'Short (D1)', 2: 'Medium (D2)', 3: 'Long (D3)'}[scale]
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(QUANTILES, qqr_avg,  'r-o', markersize=4, linewidth=1.4,
                label='QQR avg (all stocks, averaged over τ)')
        ax.plot(QUANTILES, qr_betas, 'b-s', markersize=4, linewidth=1.4,
                label='Pooled QR (all stocks)')
        ax.axhline(0, linestyle='--', color='gray', linewidth=0.8)
        ax.fill_between(QUANTILES, qqr_avg, qr_betas, alpha=0.12, color='purple',
                        label='Divergence (QQR − QR)')
        ax.set_xlabel('Gold Return Quantile (θ)', fontsize=9)
        ax.set_ylabel('β coefficient', fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        print(f"  [Robustness] Average all stocks — Scale {scale}")

    for scale in SCALES:
        fig = qr_vs_qqr_plot('^NSEI', scale, beta_store, scale_data, QUANTILES)
        if fig:
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            print(f"  [Robustness] NIFTY — Scale {scale}")

    for asset in [s for s in KEY_STOCKS if s != '^NSEI']:
        for scale in SCALES:
            fig = qr_vs_qqr_plot(asset, scale, beta_store, scale_data, QUANTILES)
            if fig:
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
        print(f"  [Robustness] {asset} — all scales")

    for scale in SCALES:
        for asset in final_stocks:
            if (scale, asset) not in beta_store:
                continue
            bm      = np.array(beta_store[(scale, asset)], dtype=float)
            qqr_avg = np.nanmean(bm, axis=1)

            df_s    = scale_data[scale]
            if asset not in df_s.columns:
                continue
            stock_arr = df_s[asset].values.astype(float)
            gold_arr  = df_s['Gold'].values.astype(float)
            mask      = np.isfinite(stock_arr) & np.isfinite(gold_arr)
            if mask.sum() < 30:
                continue

            X        = sm.add_constant(stock_arr[mask])
            qr_model = QuantReg(gold_arr[mask], X)
            qr_betas = []
            for q in QUANTILES:
                try:
                    res = qr_model.fit(q=float(q), max_iter=2000)
                    qr_betas.append(res.params[1])
                except Exception:
                    qr_betas.append(np.nan)
            qr_betas = np.array(qr_betas)

            divergence = qqr_avg - qr_betas
            robustness_rows.append({
                'Asset'            : asset,
                'Scale'            : scale,
                'QQR_avg_mean'     : float(np.nanmean(qqr_avg)),
                'QR_mean'          : float(np.nanmean(qr_betas)),
                'Divergence_mean'  : float(np.nanmean(divergence)),
                'Divergence_lowQ'  : float(np.nanmean(divergence[:5])),   # theta 0.05-0.25
                'Divergence_highQ' : float(np.nanmean(divergence[-5:])),  # theta 0.75-0.95
                'Max_abs_divergence': float(np.nanmax(np.abs(divergence)))
            })

print(f"Robustness PDF saved to: {robustness_pdf}")

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(os.path.join(OUTPUT_DIR, 'QR_QQR_Divergence_Stats.csv'), index=False)

with pd.ExcelWriter(combined_xlsx, engine='openpyxl',
                    mode='a', if_sheet_exists='replace') as writer:
    robustness_df.to_excel(writer, sheet_name='QR_QQR_Robustness', index=False)

print("Robustness divergence stats saved to CSV + Excel sheet 'QR_QQR_Robustness'.")
print(f"\nMedian absolute divergence across all assets/scales: "
      f"{robustness_df['Max_abs_divergence'].median():.4f}")
print("\nTop 5 assets with largest QQR-QR divergence (tail sensitivity):")
print(robustness_df.nlargest(5, 'Max_abs_divergence')
      [['Asset','Scale','Divergence_lowQ','Divergence_highQ','Max_abs_divergence']]
      .to_string(index=False))


In [ ]:
print("\n" + "=" * 60)
print("ALL OUTPUTS SAVED TO:", OUTPUT_DIR)
print("Combined Excel workbook:", combined_xlsx)
print("=" * 60)
print("\nScale-wise datasets ready:")
print("  scale_data[1] -> short horizon  (D1)")
print("  scale_data[2] -> medium horizon (D2)")
print("  scale_data[3] -> long horizon   (D3)")
print("\nREADY FOR FINAL REPORT TABLES AND FIGURES.")

import zipfile
from IPython.display import FileLink

def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.endswith('.zip'):
                    continue
                file_path = os.path.join(root, file)
                arcname   = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)

output_zip = '/kaggle/working/Gold_Project_Final_Results.zip'
print("\nZipping outputs...")
zip_folder(OUTPUT_DIR, output_zip)
print("Done!")
FileLink(r'Gold_Project_Final_Results.zip')


In [ ]:
import matplotlib.ticker as mticker

thresholds   = [-0.005, -0.010, -0.020]
scale_labels = ['Scale 1\n(Short)', 'Scale 2\n(Medium)', 'Scale 3\n(Long)']
scale_ids    = [1, 2, 3]

rows = []
for scale in scale_ids:
    df_s = classification_df[classification_df['Scale'] == scale].copy()
    # exclude ^NSEI / Market_Index so counts match paper (39 equities + gold excluded)
    n    = 40
    row  = {'Scale': f'Scale {scale}'}
    for thr in thresholds:
        count = (df_s['Crash_Beta'] < thr).sum()
        row[f'beta < {thr}'] = f'{count}  ({100*count/n:.1f}%)'
    rows.append(row)

df_sensitivity = pd.DataFrame(rows).set_index('Scale')
print('=== Threshold Sensitivity: Safe Haven Classification ===')
print(df_sensitivity.to_string())
df_sensitivity.to_csv(os.path.join(OUTPUT_DIR, 'Threshold_Sensitivity.csv'))

colors  = ['#2166ac', '#f4a582', '#d6604d']
markers = ['o', 's', '^']
x       = range(3)

linestyles = ['-.', '--', '-']
fig, ax = plt.subplots(figsize=(7, 4), dpi=300)
for i, thr in enumerate(thresholds):
    pcts = []
    for scale in scale_ids:
        df_s = classification_df[classification_df['Scale'] == scale]
        n    = 40
        pcts.append(100 * (df_s['Crash_Beta'] < thr).sum() / n)
    ax.plot(list(x), pcts, marker=markers[i], color=colors[i],linewidth=1.8, markersize=7, linestyle=linestyles[i],label=f'$\\beta$ < {thr}')

ax.set_xticks(list(x))
ax.set_xticklabels(scale_labels)
ax.set_ylabel('Safe Haven Probability (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
ax.legend(title='Threshold', frameon=True)
ax.grid(True, alpha=0.3)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'threshold_sensitivity.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:

EVENTS = [
    ('2016-11-08', '2016-12-31', 'Demonetisation'),
    ('2020-02-15', '2020-05-31', 'COVID-19'),
    ('2022-02-24', '2022-06-30', 'Russia–Ukraine'),
]

# 63-day (~3 month) rolling std as a volatility proxy
gold_ret  = data['Gold']
nsei_ret  = data['^NSEI'] if '^NSEI' in data.columns else None

gold_vol  = gold_ret.rolling(63).std() * np.sqrt(252)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), dpi=300,
                          sharex=True,
                          gridspec_kw={'height_ratios': [2, 2, 1.2]})


ax0 = axes[0]
ax0.plot(gold_ret.index, gold_ret.values,
         color='#c9a227', linewidth=0.6, alpha=0.85)
ax0.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax0.set_ylabel('Gold Log Return')


ax1 = axes[1]
if nsei_ret is not None:
    ax1.plot(nsei_ret.index, nsei_ret.values,
             color='#2166ac', linewidth=0.6, alpha=0.85)
    ax1.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax1.set_ylabel('NIFTY Log Return')
else:
    ax1.text(0.5, 0.5, 'NIFTY not in dataset',
             transform=ax1.transAxes, ha='center')
    ax1.set_ylabel('NIFTY Log Return')


ax2 = axes[2]
ax2.fill_between(gold_vol.index, gold_vol.values,
                  color='#c9a227', alpha=0.45)
ax2.plot(gold_vol.index, gold_vol.values,
         color='#c9a227', linewidth=0.8)
ax2.set_ylabel('Gold Ann.\nVol. (63d)')
ax2.set_xlabel('Date')

# ── Shade crisis windows across all panels ────────────────────
for ev_start, ev_end, ev_label in EVENTS:
    for ax in axes:
        ax.axvspan(pd.Timestamp(ev_start), pd.Timestamp(ev_end),
                   alpha=0.18, color='#d73027', zorder=0)
    # annotate on top panel only
    mid = pd.Timestamp(ev_start) + \
          (pd.Timestamp(ev_end) - pd.Timestamp(ev_start)) / 2
    ymax = axes[0].get_ylim()[1]
    axes[0].text(mid, ymax * 0.88, ev_label,
                 ha='center', va='top', fontsize=7.5,
                 color='#7f0000', style='italic', fontweight='bold')

for ax in axes:
    ax.grid(True, alpha=0.25)
fig.patch.set_facecolor('white')
for ax in axes:
    ax.set_facecolor('white')
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'crisis_event_returns.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')
